# 🤖 RAG Chatbot over Financial Regulations (Basel III / SEC Filings)

**AI Engineer Track**  |  Difficulty: **Hard**  |  Domain: **Regulatory Compliance / RiskTech**

> 💯 Built with 100% free tools — no paid API keys required. Uses a free `call_llm()` helper that
> tries **Cerebras → Groq → local Ollama** in that order, so this notebook costs $0 to run.

---

## 🧩 Problem Statement

Compliance teams need fast, accurate answers to regulation questions, but can't paste confidential regulatory text into a paid third-party API. Build a fully local Retrieval-Augmented Generation chatbot: free embeddings, a free vector index, and a free local LLM -- so regulatory text never leaves the machine and there is no API bill.

## 📁 Dataset

**Public Basel III framework PDFs or SEC EDGAR filings (10-K text)**

Source: [https://www.bis.org/bcbs/basel3.htm](https://www.bis.org/bcbs/basel3.htm)

⚠️ **Note:** If the real dataset file isn't uploaded to this Colab session, the code below
automatically generates a small realistic sample dataset with the same structure — so every cell
still runs successfully end-to-end even before you upload the real data.

## 🛠️ Tools Used

`Python 3 | sentence-transformers | FAISS | Cerebras/Groq (free tiers) + Ollama fallback via local HTTP (all free, no paid API) | pdfplumber`


### ⚠️ Disclaimer
This notebook is for educational / portfolio purposes only. It does not constitute financial,
legal, or investment advice.

---


In [3]:
# pillow 12.0.0 has a known broken release that crashes on import in Colab (Nov 2025 bug)
# so we pin it to stay below that version specifically, not just "latest"
!pip uninstall -y pillow -q
!pip install --no-cache-dir "pillow>=10.4,<12" sentence-transformers faiss-cpu openai pdfplumber --break-system-packages -q
print("done installing - if you still see a Pillow/_Ink import error below, go to")
print("Runtime > Restart session once, then just re-run this notebook from the top")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.6 MB/s eta 0:00:00
done installing - if you still see a Pillow/_Ink import error below, go to
Runtime > Restart session once, then just re-run this notebook from the top


In [4]:
import os
from getpass import getpass
import requests
from openai import OpenAI
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# STEP 0: set up our 3 free AI options - cerebras first, then groq, then local ollama
# we ask for the api keys once, then build one call_llm() function that tries all 3
# ---------------------------------------------------------
if not os.environ.get("CEREBRAS_API_KEY"):
    os.environ["CEREBRAS_API_KEY"] = getpass("Enter your Cerebras API key (press enter to skip): ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key (press enter to skip): ")

cerebras_client = OpenAI(api_key=os.environ.get("CEREBRAS_API_KEY", ""), base_url="https://api.cerebras.ai/v1")
groq_client = OpenAI(api_key=os.environ.get("GROQ_API_KEY", ""), base_url="https://api.groq.com/openai/v1")
OLLAMA_URL = "http://localhost:11434/api/generate"

def call_llm(prompt, system=None):
    # if there's a system instruction, we just stick it on top of the prompt
    # since we're keeping this simple and not building a full messages list
    full_prompt = f"{system}\n\n{prompt}" if system else prompt
    messages = [{"role": "user", "content": full_prompt}]

    # try cerebras first, it's free and fast
    try:
        response = cerebras_client.chat.completions.create(model="gpt-oss-120b", messages=messages)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("cerebras failed:", e)

    # try groq next
    try:
        response = groq_client.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("groq failed:", e)

    # last resort - ollama running locally, called with a plain http request
    # (no extra "ollama" package needed, just "requests" which colab already has)
    try:
        resp = requests.post(OLLAMA_URL, json={"model": "llama3", "prompt": full_prompt, "stream": False})
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print("ollama failed too:", e)
        return "AI call failed, all 3 options did not work"


In [5]:


# ---------------------------------------------------------
# STEP 1: load the regulation pdf, and if it's missing just use some
# sample regulation-style text so the rest of the code has something to work with
# ---------------------------------------------------------
def load_regulation_text(path="basel_iii_framework.pdf"):
    if os.path.exists(path):
        import pdfplumber
        text = ""
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text += (page.extract_text() or "") + "\n"
        return text
    print(f"couldn't find {path}, using some sample regulation text instead so the code still runs")
    return """Section 1: Minimum Capital Requirements. Banks must maintain a Common Equity Tier 1
capital ratio of at least 4.5% of risk weighted assets at all times.
Section 2: Liquidity Coverage Ratio. Banks must hold enough high quality liquid assets
to cover 30 days of net cash outflow under a stress scenario.
Section 3: Leverage Ratio. Banks must maintain a minimum leverage ratio of 3%, calculated
as Tier 1 capital divided by total exposure.
Section 4: Countercyclical Buffer. Regulators may require an additional buffer of up to
2.5% of risk weighted assets during periods of excessive credit growth."""


raw_text = load_regulation_text()

# split the text into small overlapping pieces, this is standard for RAG
def chunk_text(text, chunk_words=300, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + chunk_words]))
        i += chunk_words - overlap
    return chunks

chunks = chunk_text(raw_text)
print(f"made {len(chunks)} chunks from the regulation doc")

couldn't find basel_iii_framework.pdf, using some sample regulation text instead so the code still runs
made 1 chunks from the regulation doc


In [6]:
# ---------------------------------------------------------
# STEP 2: turn each chunk into a vector (free, runs on your own cpu)
# ---------------------------------------------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# ---------------------------------------------------------
# STEP 3: put all the vectors into a FAISS index so we can search them fast
# ---------------------------------------------------------
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(chunk_embeddings).astype("float32"))

In [8]:
# ---------------------------------------------------------
# STEP 4: given a question, grab the most relevant chunks
# ---------------------------------------------------------
def retrieve(question, k=4):
    q_vec = embedder.encode([question]).astype("float32")
    distances, indices = index.search(q_vec, k)
    return [chunks[i] for i in indices[0]]

In [9]:
# ---------------------------------------------------------
# STEP 5: give the retrieved chunks + question to our free call_llm() and get an answer back
# ---------------------------------------------------------
def rag_answer(question):
    retrieved = retrieve(question)
    context = "\n\n---\n\n".join(retrieved)
    prompt = f"""You're a compliance assistant. Answer the question using ONLY the context
below, nothing outside of it. If it's not in the context just say you don't know.
Mention which excerpt number backs up your answer.

Context:
{context}

Question: {question}"""
    return call_llm(prompt), retrieved

# simple function to ask a question and print the answer
def ask(question):
    answer, sources = rag_answer(question)
    print(f"Q: {question}\n")
    print(f"A: {answer}\n")
    print(f"[used {len(sources)} chunks to answer this]")
    return answer

if __name__ == "__main__":
    ask("What is the minimum common equity tier 1 capital ratio under Basel III?")


cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
Q: What is the minimum common equity tier 1 capital ratio under Basel III?

A: The minimum Common Equity Tier 1 (CET1) capital ratio is **4.5% of risk‑weighted assets**. This is stated in **Excerpt 1** (Section 1: Minimum Capital Requirements).

[used 4 chunks to answer this]
